First we want to collect 10K random sentences in COCA.

In [1]:
#from __future__ import print_function
#import time
#import numpy as np
import argparse
import pandas as pd
import numpy as np
import random

#from sklearn.decomposition import PCA
#from sklearn.manifold import TSNE
import pyarrow
import fastparquet

import csv

import spacy
from collections import defaultdict 
from tqdm import tqdm

/home/gsc685/.conda/envs/acl_metapragmatics/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

_COCA_ARCHIVE_PATH = '/home/gsc685/data/coca.2017.parquet'
df = pd.read_parquet(_COCA_ARCHIVE_PATH, engine='pyarrow')

In [7]:
df.head()

,textID,doc_text,#words,year,genre,subgen,source,title
0,170001,Headnote # Abstract : Prior research on corpor...,NaN,NaN,None,NaN,None,None
1,170002,Headnote # Abstract Consumers have dissimilar ...,NaN,NaN,None,NaN,None,None
2,170004,Headnote # ABSTRACT # The central thrust of th...,NaN,NaN,None,NaN,None,None
3,170005,Headnote # Abstract # As revenue represents on...,NaN,NaN,None,NaN,None,None
4,170006,Headnote # ABSTRACT Most available estimates o...,NaN,NaN,None,NaN,None,None


remember that this is the cleaned COCA corpus. Even though its called coca.2017 it was made using all the text files using the clean_coca.py script. 
for some reason the metadate wasnt grabbed for all of the docs but thats something we might have to wait to fix until later. 

for now, we want to sample 10k sentences. We'll build a while loop, jumping around random docs, and then sample sentences at random from in each doc.
To make things slightly easier for us down the line we'll store a different row for each word in the sentence

We want to store the following info:

token_id    word    sentence POS dependency label

In [7]:

# Set target number of sentences
TARGET_SENTENCES = 10_000
collected_sentences = 0

# load spacy pipeline
# Load the English pipeline
nlp = spacy.load("en_core_web_sm")

# Shuffle document indices
doc_indices = list(df.index)
random.shuffle(doc_indices)

# create data structure for results
data = []

# Loop until we gather 10k sentences
i = 0
token_id = 0
with tqdm(total=TARGET_SENTENCES) as pbar:
    while collected_sentences < TARGET_SENTENCES and i < len(doc_indices):
        # Get a random document
        doc_text = df.loc[doc_indices[i], 'doc_text']
        i += 1

        # Use spaCy to process the document
        doc = nlp(doc_text)
        sentences = list(doc.sents)


        # Filter sentences and shuffle
        condition = lambda sent: len(sent) < 300 and len(sent.split()) > 5
        valid_sents = [sent for sent in doc.sents if condition(sent.text)]


        sampled_sents = random.sample(valid_sents, min(15, len(valid_sents)))
        random.shuffle(sentences)

        for sent in sampled_sents:
            if collected_sentences >= TARGET_SENTENCES:
                break

            for token in sent:
                token_id +=1

                data.append({
                    'token_id': token_id,
                    'word': token.text,
                    'sentence': sent.text,
                    'pos_coarse': token.pos_,
                    'pos_fine': token.tag_,
                    'dependency': token.dep_,
                    'lemma': token.lemma_,
                    'head': token.head.text,
                    'head_pos': token.head.pos_,
                    'head_lemma': token.head.lemma_,
                    'head_dependency': token.head.dep_,


                    # NER
                    'entity_type': token.ent_type_,
                    'entity_label': token.ent_iob_,

                    # Linguistic features
                    'is_alpha': token.is_alpha,
                    'is_stop': token.is_stop,
                    'is_punct': token.is_punct,
                    'shape': token.shape_,
                })

            collected_sentences += 1
            pbar.update(1)

# Convert to DataFrame
result_df = pd.DataFrame(data)

# remove punctuation
# don't consider punctuation
result_df = result_df[result_df['pos_coarse'] != 'PUNCT']
result_df = result_df[result_df['pos_coarse'] != 'SYM']

# roberta hates dashes
result_df = result_df[~result_df['lemma'].str.contains('-')]

# limit to 1000 samples per lemma
result_df = result_df.groupby("lemma", group_keys=False).apply(lambda x: x.sample(n=min(1000, len(x))))

# dont consider words with extrmely low frequency
result_df = result_df.groupby('lemma').filter(lambda x: len(x) > 5)

# Save to CSV
result_df.to_csv("/home/gsc685/data/coca_sampled_tokenized_sentences.csv", index=False)
print("Saved 10k tokenized sentences.")

100%|██████████| 10000/10000 [01:55<00:00, 86.92it/s]
/tmp/ipykernel_147355/2727220005.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = result_df.groupby("lemma", group_keys=False).apply(lambda x: x.sample(n=min(1000, len(x))))


Saved 10k tokenized sentences.


In [8]:
result_df

,token_id,word,sentence,pos_coarse,pos_fine,dependency,lemma,head,head_pos,head_lemma,head_dependency,entity_type,entity_label,is_alpha,is_stop,is_punct,shape
136537,136538,#,"# ' I could meet you there , ' he said , again...",NOUN,NNS,npadvmod,#,meet,VERB,meet,ccomp,,O,False,False,True,#
214807,214808,#,"The season 's rich texturessuede , snakeskin ,...",NOUN,NN,dobj,#,bags,VERB,bag,ccomp,,O,False,False,True,#
181672,181673,#,Post-Its # YOU BE THE JUDGE #,NOUN,NN,dep,#,BE,VERB,be,ROOT,,O,False,False,True,#
43570,43571,#,"# But Osato lay awake that night , gazing thro...",NOUN,NNS,dep,#,lay,VERB,lie,ROOT,,O,False,False,True,#
68389,68390,#,# larisha did n't know what to make of their c...,NOUN,NN,dep,#,know,VERB,know,ROOT,,O,False,False,True,#
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40398,40399,zones,They talk about experiences in their respectiv...,NOUN,NNS,pobj,zone,in,ADP,in,prep,,O,True,False,False,xxxx
8772,8773,Zones,Cardinal flower ( Lobelia cardinalis ; Zones 3...,NOUN,NNS,nsubj,zone,is,AUX,be,ROOT,,O,True,False,False,Xxxxx
11660,11661,zone,"On Afghanistan , Carter said he backed Obama '...",NOUN,NN,pobj,zone,from,ADP,from,prep,,O,True,False,False,xxxx
31009,31010,zone,Byron poked the puck away from Matt Irwin in t...,NOUN,NN,pobj,zone,in,ADP,in,prep,,O,True,False,False,xxxx


In [10]:
df = pd.read_csv("/home/gsc685/data/coca_sampled_tokenized_sentences.csv")
df[df['word'] == 'you']

,token_id,word,sentence,pos_coarse,pos_fine,dependency,lemma,head,head_pos,head_lemma,head_dependency,entity_type,entity_label,is_alpha,is_stop,is_punct,shape
558,559,you,By continuing to browse the site you are agree...,PRON,PRP,nsubj,you,agreeing,VERB,agree,relcl,NaN,O,True,True,False,xxx
1266,1267,you,So what did you say to him ?,PRON,PRP,nsubj,you,say,VERB,say,ROOT,NaN,O,True,True,False,xxx
1279,1280,you,"This Week with George Stephanopoulos "" brought...",PRON,PRP,pobj,you,to,ADP,to,prep,NaN,O,True,True,False,xxx
1307,1308,you,"Mr. Gray , you are a local attorney .",PRON,PRP,nsubj,you,are,AUX,be,ROOT,NaN,O,True,True,False,xxx
1314,1315,you,But you have to communicate that .,PRON,PRP,nsubj,you,have,VERB,have,ROOT,NaN,O,True,True,False,xxx
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218724,218725,you,"You may not get exactly what you want , but yo...",PRON,PRP,nsubj,you,get,VERB,get,conj,NaN,O,True,True,False,xxx
218731,218732,you,"You may not get exactly what you want , but yo...",PRON,PRP,nsubj,you,paid,VERB,pay,advcl,NaN,O,True,True,False,xxx
218824,218825,you,If you have any questions or need further info...,PRON,PRP,nsubj,you,have,VERB,have,advcl,NaN,O,True,True,False,xxx
218976,218977,you,If you are not an Omaha World-Herald Subscribe...,PRON,PRP,nsubj,you,are,AUX,be,advcl,NaN,O,True,True,False,xxx


In [11]:
df.pos_coarse.value_counts()

pos_coarse
NOUN     41050
PUNCT    28754
VERB     23155
ADP      21684
PROPN    19015
DET      17325
PRON     14047
ADJ      13492
AUX      10321
ADV       7490
CCONJ     6277
PART      5740
NUM       4571
SCONJ     3577
X         2011
SYM        553
INTJ       228
Name: count, dtype: int64

In [12]:
df.groupby('pos_coarse')['word'].nunique()

pos_coarse
ADJ      2906
ADP       143
ADV       820
AUX        98
CCONJ      31
DET        58
INTJ       57
NOUN     9099
NUM       950
PART       15
PRON      138
PROPN    7403
PUNCT      46
SCONJ      70
SYM        10
VERB     4901
X          95
Name: word, dtype: int64

In [13]:
df[df['pos_coarse'] != 'PUNCT']['lemma'].nunique()

19378

In [14]:

#how many sentences are html tags?
df[df['sentence'].str.contains('<')]

,token_id,word,sentence,pos_coarse,pos_fine,dependency,lemma,head,head_pos,head_lemma,head_dependency,entity_type,entity_label,is_alpha,is_stop,is_punct,shape
436,437,<,"<p> Designed as a secure , cloud-based or on-p...",X,XX,nmod,<,>,X,>,dep,NaN,O,False,False,False,<
437,438,p,"<p> Designed as a secure , cloud-based or on-p...",X,XX,nmod,p,>,X,>,dep,NaN,O,True,False,False,x
438,439,>,"<p> Designed as a secure , cloud-based or on-p...",X,ADD,dep,>,gives,VERB,give,ROOT,NaN,O,False,False,False,>
439,440,Designed,"<p> Designed as a secure , cloud-based or on-p...",VERB,VBN,acl,design,>,X,>,dep,NaN,O,True,False,False,Xxxxx
440,441,as,"<p> Designed as a secure , cloud-based or on-p...",ADP,IN,prep,as,Designed,VERB,design,acl,NaN,O,True,True,False,xx
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217357,217358,-,"<h> Tewksbury male , 18 , arrested for assault...",NOUN,NN,pobj,-,via,ADP,via,prep,NaN,O,False,False,True,-
217358,217359,mail,"<h> Tewksbury male , 18 , arrested for assault...",NOUN,NN,nmod,mail,p,X,p,pobj,NaN,O,True,False,False,xxxx
217359,217360,<,"<h> Tewksbury male , 18 , arrested for assault...",X,XX,nmod,<,p,X,p,pobj,NaN,O,False,False,False,<
217360,217361,p,"<h> Tewksbury male , 18 , arrested for assault...",X,XX,pobj,p,via,ADP,via,prep,NaN,O,True,False,False,x


In [15]:
# don't consider punctuation
sampled_df = df[df['pos_coarse'] != 'PUNCT']
sampled_df = sampled_df[sampled_df['pos_coarse'] != 'SYM']

# dont consider words with extrmely low frequency
sampled_df = sampled_df.groupby('lemma').filter(lambda x: len(x) > 10)

In [16]:
lemmas = sampled_df['lemma'].value_counts().reset_index()


# Assume df is your DataFrame
n_samples = 1000
total_rows =len(lemmas)

# Generate 1000 evenly spaced indices
indices = np.linspace(0, len(lemmas) - 1, n_samples, dtype=int)

# Select those rows
sampled_lemmas = lemmas.iloc[indices].reset_index(drop=True)

In [17]:
sampled_lemmas['lemma'].value_counts().reset_index()


,lemma,count
0,the,1
1,Mary,1
2,instance,1
3,stock,1
4,football,1
...,...,...
995,dead,1
996,answer,1
997,11,1
998,per,1


In [18]:
# get data for sampled lemmas

sampled_df = sampled_df[sampled_df['lemma'].isin(sampled_lemmas['lemma'])]

In [19]:
sampled_df['pos_coarse'].value_counts()

pos_coarse
DET      15744
NOUN     15444
AUX       8379
PRON      7857
VERB      7320
ADP       5680
CCONJ     5573
ADJ       4468
SCONJ     3090
PROPN     2881
ADV       2578
NUM       1945
PART      1449
X          662
INTJ       118
Name: count, dtype: int64

In [20]:
sampled_df.groupby('pos_coarse')['lemma'].nunique()

pos_coarse
ADJ      166
ADP       26
ADV       85
AUX       15
CCONJ      7
DET       17
INTJ       8
NOUN     586
NUM       46
PART       2
PRON      28
PROPN    162
SCONJ     19
VERB     322
X          5
Name: lemma, dtype: int64

In [21]:
sampled_df = (
    sampled_df.groupby('lemma', group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))
)

/tmp/ipykernel_2150963/3084748779.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))


In [22]:
# finally, remove dashes
sampled_df = sampled_df[~sampled_df['lemma'].str.contains('-')]

In [23]:
sampled_df

,token_id,word,sentence,pos_coarse,pos_fine,dependency,lemma,head,head_pos,head_lemma,head_dependency,entity_type,entity_label,is_alpha,is_stop,is_punct,shape
103671,103672,#,# ESSENCE : Why do you think your march had su...,NOUN,NN,compound,#,ESSENCE,NOUN,essence,dep,NaN,O,False,False,True,#
127576,127577,#,"# SINCERELY , TOM DELAY # Dear Mr. Staggs # AN...",NOUN,NN,ROOT,#,#,NOUN,#,ROOT,NaN,O,False,False,True,#
109372,109373,#,# Silver touched the necklace beneath the coll...,NOUN,NN,dep,#,touched,VERB,touch,ROOT,NaN,O,False,False,True,#
44689,44690,#,"Hi Joe , # I 'm planning a moderately priced g...",NOUN,NN,npadvmod,#,planning,VERB,plan,ROOT,NaN,O,False,False,True,#
197424,197425,#,shaft lengths # PROS # DISTANCE CONTROL :,NOUN,NN,compound,#,DISTANCE,NOUN,distance,compound,NaN,O,False,False,True,#
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88309,88310,young,I mean the only problem with the children of R...,ADJ,JJ,amod,young,candidate,NOUN,candidate,attr,NaN,O,True,False,False,xxxx
175790,175791,younger,"On a variety of measures , the older prephonol...",ADJ,JJR,amod,young,spellers,NOUN,speller,nsubj,NaN,O,True,False,False,xxxx
202758,202759,young,He then brought the young girl to his bedroom ...,ADJ,JJ,amod,young,girl,NOUN,girl,dobj,NaN,O,True,False,False,xxxx
65086,65087,young,Certainly there are plenty of young men and wo...,ADJ,JJ,amod,young,men,NOUN,man,pobj,NaN,O,True,False,False,xxxx


In [ ]:
sampled_df.to_csv("/home/gsc685/data/coca_downsampled_tokenized_sentences.csv", index=False)

In [ ]:
#isample sentences again but without tokenizing, for looking at entire sentences


In [ ]:

# load coca
_COCA_ARCHIVE_PATH = '/home/gsc685/data/coca.2017.parquet'
df = pd.read_parquet(_COCA_ARCHIVE_PATH, engine='pyarrow')

# Set target number of sentences
TARGET_SENTENCES = 5_000
collected_sentences = 0

# load spacy pipeline
# Load the English pipeline
nlp = spacy.load("en_core_web_sm")

# Shuffle document indices
doc_indices = list(df.index)
random.shuffle(doc_indices)

# create data structure for results
sampled_sentences = []

# Loop until we gather 10k sentences
i = 0
token_id = 0
with tqdm(total=TARGET_SENTENCES) as pbar:
    while collected_sentences < TARGET_SENTENCES and i < len(doc_indices):
        # Get a random document
        doc_text = df.loc[doc_indices[i], 'doc_text']

        # Use spaCy to process the document
        doc = nlp(doc_text)
        sentences = list(doc.sents)


        # Filter sentences and shuffle
        condition = lambda sent: len(sent) < 300 and len(sent.split()) > 5
        valid_sents = [sent for sent in doc.sents if condition(sent.text)]


        sampled_sents = random.sample(valid_sents, min(5, len(valid_sents)))
        random.shuffle(sentences)

        # get five random sentences from this document
        for sent in sampled_sents:
            if collected_sentences >= TARGET_SENTENCES:
                break
            sampled_sentences.append({
                'sentence': sent.text,
                'sentence_id': i,
                'doc_id': doc_indices[i],
            })
            collected_sentences += 1
            pbar.update(1)
        i += 1

# Convert to DataFrame
result_df = pd.DataFrame(sampled_sentences)

# Save to CSV
result_df.to_csv("/home/gsc685/data/coca_sampled_sentences.csv", index=False)
print("Saved 10k tokenized sentences.")

100%|██████████| 5000/5000 [02:35<00:00, 32.13it/s]


NameError: name 'data' is not defined